In [88]:
import os
import glob
import numpy as np
import rasterio
import datetime as dt
from dateutil import tz
import sys
sys.path.append(r"C:\Users\SamanthaLavender\Documents\GitHub\sen2like\sen2like\sen2like")
from atmcor.atmospheric_parameters import ATMO_parameter
from atmcor.cams_data_reader import ECMWF_Product
from atmcor.smac import smac
sys.path.append(r"C:\Users\SamanthaLavender\Documents\GitHub\sen2like\prisma4sen2like\prisma")
from sunpy.coordinates import sun
from spectral_aggregation_functions import radiance_to_reflectance, sun_earth_correction


In [89]:
inpath = r"C:\DDrive\Vega\MOS\Radiometry"
tds = r"TDS1"
filename = r"MO01_MES_GEC_1P_19900619T093023_19900619T093041_MTI_16958_1858.TIFF"
infolder = os.path.join(inpath, tds, filename)
files = glob.glob(os.path.join(infolder,"MO*_B*.TIF"))
print("Found: {}".format(files))

Found: ['C:\\DDrive\\Vega\\MOS\\Radiometry\\TDS1\\MO01_MES_GEC_1P_19900619T093023_19900619T093041_MTI_16958_1858.TIFF\\MO01_MES_GEC_1P_19900619T093023_19900619T093041_MTI_16958_1858_B1.TIF', 'C:\\DDrive\\Vega\\MOS\\Radiometry\\TDS1\\MO01_MES_GEC_1P_19900619T093023_19900619T093041_MTI_16958_1858.TIFF\\MO01_MES_GEC_1P_19900619T093023_19900619T093041_MTI_16958_1858_B2.TIF', 'C:\\DDrive\\Vega\\MOS\\Radiometry\\TDS1\\MO01_MES_GEC_1P_19900619T093023_19900619T093041_MTI_16958_1858.TIFF\\MO01_MES_GEC_1P_19900619T093023_19900619T093041_MTI_16958_1858_B3.TIF', 'C:\\DDrive\\Vega\\MOS\\Radiometry\\TDS1\\MO01_MES_GEC_1P_19900619T093023_19900619T093041_MTI_16958_1858.TIFF\\MO01_MES_GEC_1P_19900619T093023_19900619T093041_MTI_16958_1858_B4.TIF']


In [90]:
# Load band data
nbands = len(files)
bands = ['560','660','860','1370']
offset = [0.0, 0.0, 0.0, 0.0]
slope = [1.0, 1.0, 1.0, 1.0]
for i,file in enumerate(files):
    ds = rasterio.open(file)
    if i == 0:
        print("Dataset: {}".format(ds.shape))
        img = np.zeros((nbands,ds.shape[0],ds.shape[1]), dtype = np.float32)
    img[i,:,:] = ds.read().astype(np.float32) * slope[i] + offset[i]
    ds.close()
print("Read in data and applied scaling to radiance: {} {}".format(np.nanmin(img),np.nanmax(img)))

Dataset: (2439, 2351)
Read in data and applied scaling to radiance: 0.0 64.0


In [91]:
# Solar zenith and azimuth angles
theta_s = 0
phi_s = 0

theta_v = 0  # View Zenith Angle Landsat (Nadir)
phi_v = 0  # View Azimuth Angle Landsat (Nadir)

In [92]:
# default params
uH2O = 2.0  # Water Vapor content - unit: g.cm-2
uO3 = 0.331  # Ozone content - unit: cm-atm , 0.3 cm-atm = 300 Dobson Units
pressure = 1013.095  # Pressure - unit: hpa
taup550 = 0.2  # taup550 - unit: unitless


In [93]:
# Convert to TOA reflectance
## from radiance (W.m-2.sr-1.um-1) to reflectance (unitless)
esun = [1000., 1000., 1000., 1000.]   
start_time = dt.datetime(1990, 6, 19, 9, 30, 41).replace(tzinfo=tz.tzutc())
sun_earth_distance = sun.earth_distance(start_time).value
print("Sun Earth Distance: {:.3f}".format(sun_earth_distance))
rtoaimg = np.zeros((nbands,ds.shape[0],ds.shape[1]), dtype = np.float32)
for band in range(nbands):
    rtoaimg[band,:,:] = radiance_to_reflectance(img[band,:,:], esun[band], theta_s, sun_earth_distance)
print("TOA reflectance: {} {}".format(np.nanmin(rtoaimg),np.nanmax(rtoaimg)))

Sun Earth Distance: 1.016
TOA reflectance: 0.0 0.2075936496257782


In [94]:
def get_smac_coefficients(product, band):
    filename = "Coef_{}_{}_1.dat".format(product,band)
    if filename is None:
        return None

    # smac coefficient are in the smac package
    smac_directory = os.path.dirname(os.path.abspath(smac.__file__))
    smac_file = os.path.join(smac_directory, 'COEFS', filename)
    if os.path.exists(smac_file):
        return smac_file
    else:
        print("Could not find {}".format(smac_file))
        return None

In [95]:

# Atmospheric parameters retrieve from CAMS :
r_toa = np.arange(101) / 100.
r_surf_SMAC = np.zeros(101)


# # Apply correction
product = "LANDSAT8"
rboaimg = np.zeros((nbands,ds.shape[0],ds.shape[1]), dtype = np.float32)
for b,band in enumerate(bands):
    print("Processing band {} {}".format(b,band))
    # Load SMAC coefficients
    coef_file = get_smac_coefficients(product, band)
    print(coef_file)
    if coef_file is None:
        raise SystemExit("No smac coefficients for %s", band)

    smac_coefs = smac.coeff(coef_file)

    # Run SMAC for r_toa ranging from 0.0 to 1.0
    for i in range(101):
        r_surf_SMAC[i] = smac.smac_inv(
            r_toa[i], theta_s, phi_s,
            theta_v, phi_v, pressure,
            taup550, uO3, uH2O, smac_coefs)
    #
    # Use a polynomial fit of order 2 to fit relation between surface reflectance and TOA reflectance
    poly_coefs = np.polyfit(r_toa, r_surf_SMAC, 2, full=True)
    if b == 0:
        print("Coeffs: {}".format(poly_coefs))
    # Apply fitted relation to convert TOA reflectance to surface reflectance
    sref = poly_coefs[0][2] + poly_coefs[0][1] * rtoaimg[b,:,:] + poly_coefs[0][0] * rtoaimg[b,:,:] ** 2
    mask = (rtoaimg[b,:,:] <= 0)
    sref[mask] = 0
    rboaimg[b,:,:] = sref[:,:]
print("BOA reflectance: {} {}".format(np.nanmin(rboaimg),np.nanmax(rboaimg)))


Processing band 0 560
C:\Users\SamanthaLavender\Documents\GitHub\sen2like\sen2like\sen2like\atmcor\smac\COEFS\Coef_LANDSAT8_560_1.dat
Coeffs: (array([-0.15864536,  1.29629264, -0.05917915]), array([1.87379287e-05]), np.int32(3), array([1.64934844, 0.51943226, 0.09919601]), np.float64(2.2426505097428162e-14))
Processing band 1 660
C:\Users\SamanthaLavender\Documents\GitHub\sen2like\sen2like\sen2like\atmcor\smac\COEFS\Coef_LANDSAT8_660_1.dat
Processing band 2 860
C:\Users\SamanthaLavender\Documents\GitHub\sen2like\sen2like\sen2like\atmcor\smac\COEFS\Coef_LANDSAT8_860_1.dat
Processing band 3 1370
C:\Users\SamanthaLavender\Documents\GitHub\sen2like\sen2like\sen2like\atmcor\smac\COEFS\Coef_LANDSAT8_1370_1.dat
BOA reflectance: -0.017298854887485504 19.492334365844727
